# Import libraries

In [ ]:
!pip install underthesea
from tqdm import tqdm
import os, glob, re, random
os.environ["WANDB_DISABLED"] = "true"
import numpy as np
from scipy.optimize import linear_sum_assignment
from underthesea import sent_tokenize, word_tokenize
from sklearn.model_selection import train_test_split
import torch, torch.nn as nn
from torch.utils.data import Dataset
from collections import Counter, defaultdict
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, AutoModel
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from sklearn.metrics import confusion_matrix, classification_report

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.6/846.6 kB 18.2 MB/s eta 0:00:00


In [5]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Load dataset

In [2]:
!gdown "1G2dlilo2LZugyVcrqb9mzlQkYBYz9aOf"
!unzip "CorefDataset.zip"
!mv annotated_texts/few_shots/* ../

BASE_DIR = "/content/annotated_texts"

def get_file(base_dir, split_name):
    return glob.glob(os.path.join(base_dir, f"{split_name}_*.txt"))

def load_data(base_dir, split_name):
    file_paths = get_file(base_dir, split_name)
    data_list = []
    for path in file_paths:
        try:
            with open(path, "r", encoding="utf-8") as f:
                data_list.append(f.read().strip())
        except Exception as e:
            print(f"Error reading {path}: {e}")
    return data_list

raw_train = load_data(BASE_DIR, "train")
raw_valid = load_data(BASE_DIR, "valid")
raw_test = load_data(BASE_DIR, "test")

all_data = raw_train + raw_valid + raw_test

train, temp = train_test_split(all_data, test_size=0.3, random_state=42, shuffle=True)
val, test = train_test_split(temp, test_size=2/3, random_state=42, shuffle=True)

Downloading...
From: https://drive.google.com/uc?id=1G2dlilo2LZugyVcrqb9mzlQkYBYz9aOf
To: /content/CorefDataset.zip
100% 2.55M/2.55M [00:00<00:00, 168MB/s]
Archive:  CorefDataset.zip
   creating: raw_texts/
  inflating: __MACOSX/._raw_texts    
  inflating: raw_texts/test_non_abusive_869.txt  
  inflating: __MACOSX/raw_texts/._test_non_abusive_869.txt  
  inflating: raw_texts/test_non_abusive_841.txt  
  inflating: __MACOSX/raw_texts/._test_non_abusive_841.txt  
  inflating: raw_texts/train_non_abusive_188.txt  
  inflating: __MACOSX/raw_texts/._train_non_abusive_188.txt  
  inflating: raw_texts/test_non_abusive_855.txt  
  inflating: __MACOSX/raw_texts/._test_non_abusive_855.txt  
  inflating: raw_texts/train_non_abusive_177.txt  
  inflating: __MACOSX/raw_texts/._train_non_abusive_177.txt  
  inflating: raw_texts/train_non_abusive_163.txt  
  inflating: __MACOSX/raw_texts/._train_non_abusive_163.txt  
  inflating: raw_texts/train_non_abusive_605.txt  
  inflating: __MACOSX/raw_texts/

In [3]:
import shutil
shutil.rmtree('/content/__MACOSX')
shutil.rmtree('/content/raw_texts')

In [4]:
def clean_metadata(text):
    cleaned = re.sub(r"^(#color:.*|#tokenization.*)$", "", text, flags=re.MULTILINE|re.IGNORECASE).strip()
    return cleaned.lower()

train = [clean_metadata(t) for t in train]
val   = [clean_metadata(t) for t in val]
test  = [clean_metadata(t) for t in test]

# Preprocessing

In [6]:
def remove_M_tags(text):
    prev = None
    while prev != text:
        prev = text
        text = re.sub(r"\{M\d+\s*", "", text, flags=re.IGNORECASE).replace("}", "")
    return re.sub(r"\s+", " ", text).strip()

text = train[0]
clean_text = remove_M_tags(text)
clean_text

'bố mẹ tôi làm nông ở làng quê bắc bộ. trong nhà tôi, đàn ông ra lệnh, đàn bà phục tùng. bố mẹ thức khuya dạy sớm, tần tảo tiết kiệm nên trông già hơn tuổi. ở tuổi ngoài 50, bố mẹ có một số tích luỹ nhỏ cùng với bất động sản. tôi là chị cả, đã kết hôn hơn chục năm. chồng ở gần nhà tôi, là người đàn ông hiện đại, cởi mở, vui vẻ, lạc quan, luôn chia sẻ và gánh vác mọi khó khăn cùng tôi. tôi có người em gái thứ hai bị bệnh bẩm sinh nhưng mạnh mẽ và giàu nghị lực. cuối cùng là em trai 25 tuổi, học không tốt, không thuộc diện chơi bời nhưng ít có chí tiến thủ, mang thói gia trưởng. tôi không phải người có nhan sắc xuất chúng để có thể tích luỹ được nhiều tài sản ở tuổi 35, thế nhưng vẫn thấy hạnh phúc vì có công việc, có đủ tay chân mắt mũi, có người chồng tốt và những đứa con ngoan. từ khi tốt nghiệp tôi luôn cố gắng hết mình giúp đỡ các em học tập, rồi mua sắm trang thiết bị trong nhà do bố mẹ rất hà tiện. cuộc sống cá nhân của tôi không dư, tôi có những bệnh lý nền khi tuổi đời còn trẻ n

In [7]:
def tokenize(text, word_seg=True):
    if word_seg:
        return word_tokenize(text, format="text").split()
    return re.findall(r"\w+|[^\w\s]", text, re.UNICODE)

tokens = tokenize(clean_text)
tokens[:10]

['bố_mẹ', 'tôi', 'làm', 'nông', 'ở', 'làng', 'quê', 'bắc_bộ', '.', 'trong']

In [8]:
def compute_token_char_offsets(text, tokens):
    offsets = []
    idx = 0
    for tok in tokens:
        while idx < len(text) and text[idx].isspace():
            idx += 1
        start = idx
        end = idx + len(tok)
        offsets.append((start, end))
        idx = end
    return offsets

token_offsets = compute_token_char_offsets(clean_text, tokens)
token_offsets[:5]

[(0, 5), (6, 9), (10, 13), (14, 18), (19, 20)]

In [9]:
def extract_identity_mentions(raw_text):
    result = []
    for m in re.finditer(r"\{(m\d+)\s+([^}]+)\}", raw_text, flags=re.IGNORECASE):
        eid = m.group(1).lower()
        mention = m.group(2)

        raw_start, raw_end = m.start(2), m.end(2)

        clean_start = len(remove_M_tags(raw_text[:raw_start]))
        clean_end   = clean_start + len(mention)

        result.append((mention, eid, clean_start, clean_end))
    return result

mentions = extract_identity_mentions(text)
mentions

[('bố mẹ tôi', 'm1', 0, 9),
 ('nhà tôi', 'm2', 43, 50),
 ('đàn ông', 'm3', 52, 59),
 ('đàn bà', 'm4', 69, 75),
 ('bố mẹ', 'm1', 87, 92),
 ('bố mẹ', 'm1', 172, 177),
 ('tôi', 'm5', 224, 227),
 ('chồng', 'm6', 264, 269),
 ('tôi', 'm5', 280, 283),
 ('người đàn ông', 'm6', 288, 301),
 ('tôi', 'm5', 381, 384),
 ('tôi', 'm5', 386, 389),
 ('người em gái thứ hai', 'm7', 393, 413),
 ('em trai', 'm9', 476, 483),
 ('tôi', 'm5', 582, 585),
 ('người chồng', 'm6', 748, 759),
 ('những đứa con', 'm14', 767, 780),
 ('tôi', 'm5', 806, 809),
 ('các em', 'm17', 840, 846),
 ('bố mẹ', 'm1', 896, 901),
 ('tôi', 'm5', 937, 940),
 ('tôi', 'm5', 951, 954),
 ('tôi', 'm5', 1073, 1076),
 ('những người thân thiết', 'm25', 1094, 1116),
 ('em gái tôi', 'm7', 1143, 1153),
 ('bản thân', 'm7', 1190, 1198),
 ('em', 'm7', 1265, 1267),
 ('em', 'm7', 1304, 1306),
 ('mạnh thường quân', 'm29', 1325, 1341),
 ('mẹ tôi', 'm8', 1362, 1368),
 ('em', 'm7', 1374, 1376),
 ('em', 'm7', 1396, 1398),
 ('tôi', 'm5', 1405, 1408),
 ('em', 

In [10]:
def map_char_span_to_token_span(token_offsets, char_start, char_end):
    token_ids = []
    for i, (ts, te) in enumerate(token_offsets):
        if not (te <= char_start or ts >= char_end):
            token_ids.append(i)
    if token_ids:
        return (token_ids[0], token_ids[-1])
    return None

print(mentions[0])

_, _, cs, ce = mentions[0]
map_char_span_to_token_span(token_offsets, cs, ce)

('bố mẹ tôi', 'm1', 0, 9)


(0, 1)

In [11]:
def mentions_to_token_spans(mentions, token_offsets):
    spans = []
    for _, eid, cs, ce in mentions:
        span = map_char_span_to_token_span(token_offsets, cs, ce)
        if span is not None:
            s, e = span
            spans.append({
                "start": s,
                "end": e,
                "eid": eid
            })
    return spans
token_spans = mentions_to_token_spans(mentions, token_offsets)
token_spans[:5]

[{'start': 0, 'end': 1, 'eid': 'm1'},
 {'start': 10, 'end': 11, 'eid': 'm2'},
 {'start': 13, 'end': 13, 'eid': 'm3'},
 {'start': 16, 'end': 16, 'eid': 'm4'},
 {'start': 19, 'end': 19, 'eid': 'm1'}]

In [12]:
def build_bio_tags_from_token_spans(num_tokens, spans):
    tags = ["O"] * num_tokens

    for span in spans:
        s, e = span["start"], span["end"]
        tags[s] = "B-M"
        for i in range(s + 1, e + 1):
            tags[i] = "I-M"

    return tags
tags = build_bio_tags_from_token_spans(len(tokens), token_spans)
tags[:5]

['B-M', 'I-M', 'O', 'O', 'O']

In [ ]:
def build_bio_from_raw_text(raw_text, word_seg=True):
    # clean text
    clean_text = remove_M_tags(raw_text)
    # tokenize
    tokens = tokenize(clean_text, word_seg)
    # token offsets
    token_offsets = compute_token_char_offsets(clean_text, tokens)
    # extract mentions (char-level)
    mentions = extract_identity_mentions(raw_text)
    # map to token spans
    token_spans = mentions_to_token_spans(mentions, token_offsets)
    # build BIO
    tags = build_bio_tags_from_token_spans(len(tokens), token_spans)

    return tokens, tags, token_spans

tokens, tags, spans = build_bio_from_raw_text(text, word_seg=False)
for i, (tok, tag) in enumerate(zip(tokens, tags)):
    print(f"{i:2d} {tok:10s} {tag}")

 0 bố         B-M
 1 mẹ         I-M
 2 tôi        I-M
 3 làm        O
 4 nông       O
 5 ở          O
 6 làng       O
 7 quê        O
 8 bắc        O
 9 bộ         O
10 .          O
11 trong      O
12 nhà        B-M
13 tôi        I-M
14 ,          O
15 đàn        B-M
16 ông        I-M
17 ra         O
18 lệnh       O
19 ,          O
20 đàn        B-M
21 bà         I-M
22 phục       O
23 tùng       O
24 .          O
25 bố         B-M
26 mẹ         I-M
27 thức       O
28 khuya      O
29 dạy        O
30 sớm        O
31 ,          O
32 tần        O
33 tảo        O
34 tiết       O
35 kiệm       O
36 nên        O
37 trông      O
38 già        O
39 hơn        O
40 tuổi       O
41 .          O
42 ở          O
43 tuổi       O
44 ngoài      O
45 50         O
46 ,          O
47 bố         B-M
48 mẹ         I-M
49 có         O
50 một        O
51 số         O
52 tích       O
53 luỹ        O
54 nhỏ        O
55 cùng       O
56 với        O
57 bất        O
58 động       O
59 sản        O
60 .          

In [14]:
def bio_to_spans(bio_tags):
    spans = []
    start = None
    for i, tag in enumerate(bio_tags):
        if tag == "B-M":
            if start is not None:
                spans.append((start, i))
            start = i
        elif tag == "I-M":
            if start is None:
                start = i
        else:  # "O"
            if start is not None:
                spans.append((start, i))
                start = None
    if start is not None:
        spans.append((start, len(bio_tags)))
    return spans

# Mention detection

In [15]:
def split_into_sentences(tokens, tags, sentence_end_tokens={".","!","?"}):
    sentences = []
    cur_tokens, cur_tags = [], []

    for tok, tag in zip(tokens, tags):
        cur_tokens.append(tok)
        cur_tags.append(tag)

        if tok in sentence_end_tokens:
            sentences.append((cur_tokens, cur_tags))
            cur_tokens, cur_tags = [], []

    if cur_tokens:
        sentences.append((cur_tokens, cur_tags))

    return sentences

split_into_sentences(tokens, tags)

[(['bố', 'mẹ', 'tôi', 'làm', 'nông', 'ở', 'làng', 'quê', 'bắc', 'bộ', '.'],
  ['B-M', 'I-M', 'I-M', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']),
 (['trong',
   'nhà',
   'tôi',
   ',',
   'đàn',
   'ông',
   'ra',
   'lệnh',
   ',',
   'đàn',
   'bà',
   'phục',
   'tùng',
   '.'],
  ['O',
   'B-M',
   'I-M',
   'O',
   'B-M',
   'I-M',
   'O',
   'O',
   'O',
   'B-M',
   'I-M',
   'O',
   'O',
   'O']),
 (['bố',
   'mẹ',
   'thức',
   'khuya',
   'dạy',
   'sớm',
   ',',
   'tần',
   'tảo',
   'tiết',
   'kiệm',
   'nên',
   'trông',
   'già',
   'hơn',
   'tuổi',
   '.'],
  ['B-M',
   'I-M',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O']),
 (['ở',
   'tuổi',
   'ngoài',
   '50',
   ',',
   'bố',
   'mẹ',
   'có',
   'một',
   'số',
   'tích',
   'luỹ',
   'nhỏ',
   'cùng',
   'với',
   'bất',
   'động',
   'sản',
   '.'],
  ['O',
   'O',
   'O',
   'O',
   'O',
   'B-M',
   'I-M',
   'O',
   'O',
   'O',
   'O',
 

In [ ]:
def chunk_by_sentences(sentences, max_len=512):
    chunks = []
    cur_tokens, cur_tags = [], []

    for sent_tokens, sent_tags in sentences:

        # sentence quá dài -> fallback chunk thô
        if len(sent_tokens) > max_len:
            for i in range(0, len(sent_tokens), max_len):
                chunks.append((
                    sent_tokens[i:i+max_len],
                    sent_tags[i:i+max_len]
                ))
            continue

        if len(cur_tokens) + len(sent_tokens) <= max_len:
            cur_tokens += sent_tokens
            cur_tags   += sent_tags
        else:
            chunks.append((cur_tokens, cur_tags))
            cur_tokens = sent_tokens
            cur_tags   = sent_tags

    if cur_tokens:
        chunks.append((cur_tokens, cur_tags))

    return chunks

sentences = zip(tokens, tags)
chunks = chunk_by_sentences(sentences)
len(chunks)

7

## Encode

In [ ]:
def create_bio(dataset_split, max_len=512, word_seg=True):
    dataset = []

    for raw_text in dataset_split:
        # Build BIO at document-level
        tokens, tags, token_spans = build_bio_from_raw_text(raw_text, word_seg)
        # Split into sentences (token-level)
        sentences = split_into_sentences(tokens, tags)
        # Chunk by sentence with max_len
        chunks = chunk_by_sentences(sentences, max_len=max_len)
        # Collect
        for tok_chunk, tag_chunk in chunks:
            assert len(tok_chunk) == len(tag_chunk)
            dataset.append((tok_chunk, tag_chunk))

    return dataset

In [18]:
label2id = {"O":0,"B-M":1,"I-M":2}
id2label = {v:k for k,v in label2id.items()}

In [ ]:
def encode(tokens, tags, tokenizer):
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=256
    )

    word_ids = encoding.word_ids()
    labels = []
    prev_word_id = None

    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        elif word_id != prev_word_id:
            labels.append(label2id[tags[word_id]])
        else:
            labels.append(-100)

        prev_word_id = word_id

    encoding["labels"] = labels
    return encoding

In [20]:
def decode_tokens_and_tags(input_ids, labels, preds, tokenizer):
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    assert isinstance(labels, list)
    assert isinstance(preds, list)

    result = []
    for tok, gold, pred in zip(tokens, labels, preds):
        if gold == -100:
            continue
        result.append((tok, id2label[gold], id2label[pred]))
    return result

def print_sample(sample, max_len=120):
    print("=" * 80)
    for tok, gold, pred in sample[:max_len]:
        print(f"{tok:15s} gold={gold:4s} pred={pred:4s}")

## Dataset

In [21]:
class NERDataset(Dataset):
    def __init__(self, encodings): self.encodings = encodings
    def __len__(self): return len(self.encodings)
    def __getitem__(self, idx): return {k: torch.tensor(v) for k,v in self.encodings[idx].items()}

## Training

In [22]:
def entity_f1(y_true, y_pred):
    tp = fp = fn = 0

    for t_seq, p_seq in zip(y_true, y_pred):

        valid_idx = [i for i, t in enumerate(t_seq) if t != -100]
        t_seq = [int(t_seq[i]) for i in valid_idx]
        p_seq = [int(p_seq[i]) for i in valid_idx]

        t_tags = [id2label[t] for t in t_seq]
        p_tags = [id2label[p] for p in p_seq]

        gold_spans = set(bio_to_spans(t_tags))
        pred_spans = set(bio_to_spans(p_tags))

        tp += len(gold_spans & pred_spans)
        fp += len(pred_spans - gold_spans)
        fn += len(gold_spans - pred_spans)

    p = tp / (tp + fp + 1e-10)
    r = tp / (tp + fn + 1e-10)
    f1 = 2 * p * r / (p + r + 1e-10)
    return p, r, f1

In [23]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)

    y_true_seq = []
    y_pred_seq = []

    for p_seq, l_seq in zip(preds, labels):
        y_true_seq.append(l_seq.tolist())
        y_pred_seq.append(p_seq.tolist())

    p, r, f1 = entity_f1(y_true_seq, y_pred_seq)
    return {
        "entity_precision": p,
        "entity_recall": r,
        "entity_f1": f1
    }

In [ ]:
def run_experiment(model_name, tokenizer_name, train_bio, val_bio, test_bio,
                   lr=2e-5, epochs=5, save_dir="ner_model"):
    print(f"\n{'='*30}\nMô hình: {model_name}\n{'='*30}")

    # Tokenize
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)

    # Re-encode dữ liệu (vì subwords của mỗi mô hình mỗi khác)
    train_enc = [encode(t, tag, tokenizer) for t, tag in train_bio]
    val_enc   = [encode(t, tag, tokenizer) for t, tag in val_bio]
    test_enc  = [encode(t, tag, tokenizer) for t, tag in test_bio]

    # Khởi tạo Dataset
    train_dataset = NERDataset(train_enc)
    val_dataset   = NERDataset(val_enc)
    test_dataset  = NERDataset(test_enc)

    # Khởi tạo Model
    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id
    )

    # Cấu hình Training
    training_args = TrainingArguments(
        output_dir=save_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=8,
        num_train_epochs=epochs,
        load_best_model_at_end=True,
        metric_for_best_model="entity_f1",
        greater_is_better=True,
        report_to="none"
    )

    # Huấn luyện
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )
    trainer.train()

    print(f"Saving best NER model to {save_dir}")
    trainer.save_model(save_dir) 
    tokenizer.save_pretrained(save_dir)

    # Đánh giá
    print(f"Kết quả trên Test Set của {model_name}:")
    metrics = trainer.evaluate(test_dataset)
    print(metrics)

    return metrics

## Eval

In [ ]:
def eval(test_dataset, ner_model):
    y_true_seq = []     # cho span-level
    y_pred_seq = []

    y_true_flat = []    # cho token-level
    y_pred_flat = []

    with torch.no_grad():
        for batch in test_dataset:
            input_ids = batch["input_ids"].unsqueeze(0).to(device)
            attention_mask = batch["attention_mask"].unsqueeze(0).to(device)

            gold_labels = batch["labels"].tolist()

            outputs = ner_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            pred_labels = outputs.logits.argmax(-1).squeeze(0).cpu().tolist()

            # ----- span-level -----
            y_true_seq.append(gold_labels)
            y_pred_seq.append(pred_labels)

            # ----- token-level -----
            for g, p in zip(gold_labels, pred_labels):
                if g != -100:          # bỏ padding + subword continuation
                    y_true_flat.append(g)
                    y_pred_flat.append(p)

    # Span-level (Entity) Eval
    p, r, f1 = entity_f1(y_true_seq, y_pred_seq)

    print("=== NER Span-level Evaluation ===")
    print(f"Precision: {p:.4f}")
    print(f"Recall:    {r:.4f}")
    print(f"F1-score:  {f1:.4f}")

    # Token-level Eval
    cm = confusion_matrix(y_true_flat, y_pred_flat)
    print("\n=== Token-level Confusion Matrix ===")
    print(cm)

    labels = sorted(id2label.keys())
    target_names = [id2label[i] for i in labels]

    print("\n=== Token-level Classification Report ===")
    print(classification_report(
        y_true_flat,
        y_pred_flat,
        labels=labels,
        target_names=target_names,
        digits=4
    ))

# Result ner

#### With word segmentation

In [56]:
train_bio = create_bio(train, word_seg=True)
val_bio   = create_bio(val, word_seg=True)
test_bio  = create_bio(test, word_seg=True)

In [ ]:
model_name = "vinai/phobert-base"
tokenizer_name = "hieule/bartpho-tokenizer-fast"

res = run_experiment(model_name, tokenizer_name, train_bio, val_bio, test_bio,
                     lr=2e-5, epochs=20, save_path= "ner_phobert.pt")


Mô hình: vinai/phobert-base


Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Entity Precision,Entity Recall,Entity F1
1,No log,0.275297,0.721170,0.662975,0.690849
2,No log,0.202362,0.716923,0.737342,0.726989
3,No log,0.171371,0.717360,0.791139,0.752445
4,No log,0.153640,0.742382,0.848101,0.791728
5,No log,0.132255,0.821162,0.871835,0.845741
6,No log,0.132549,0.813783,0.878165,0.844749
7,No log,0.123289,0.849379,0.865506,0.857367
8,No log,0.126320,0.818047,0.875000,0.845566
9,No log,0.128361,0.823704,0.879747,0.850803
10,No log,0.130088,0.809104,0.871835,0.839299


Kết quả trên Test Set của vinai/phobert-base:


{'eval_loss': 0.12600968778133392, 'eval_entity_precision': 0.859594383775284, 'eval_entity_recall': 0.854926299456877, 'eval_entity_f1': 0.8572539867255075, 'eval_runtime': 0.9942, 'eval_samples_per_second': 69.401, 'eval_steps_per_second': 9.052, 'epoch': 20.0}


In [ ]:
tokenizer_name = "xlm-roberta-base"
model_name = "xlm-roberta-base"

res = run_experiment(model_name, tokenizer_name, train_bio, val_bio, test_bio,
                     lr=2e-5, epochs=10, save_path= "ner_roberta.pt")


Mô hình: xlm-roberta-base


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Entity Precision,Entity Recall,Entity F1
1,No log,0.131612,0.635383,0.802452,0.709211
2,No log,0.075340,0.902537,0.920981,0.911666
3,No log,0.073370,0.913747,0.923706,0.918699
4,No log,0.071343,0.922869,0.929155,0.926001
5,No log,0.077640,0.903694,0.933243,0.918231
6,No log,0.078935,0.908367,0.931880,0.919973
7,No log,0.079565,0.914894,0.937330,0.925976
8,No log,0.088082,0.915663,0.931880,0.923700
9,No log,0.083995,0.912351,0.935967,0.924008
10,No log,0.085425,0.917223,0.935967,0.926500


Kết quả trên Test Set của xlm-roberta-base:


{'eval_loss': 0.06988441944122314, 'eval_entity_precision': 0.9063214013708373, 'eval_entity_recall': 0.9333333333332601, 'eval_entity_f1': 0.9196290571369566, 'eval_runtime': 1.0708, 'eval_samples_per_second': 65.371, 'eval_steps_per_second': 8.405, 'epoch': 10.0}


In [ ]:
tokenizer_name = "microsoft/mdeberta-v3-base"
model_name = "microsoft/mdeberta-v3-base"

res = run_experiment(model_name, tokenizer_name, train_bio, val_bio, test_bio,
                     lr=5e-5, epochs=10, save_dir="ner_model")


Mô hình: microsoft/mdeberta-v3-base


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/mdeberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Entity Precision,Entity Recall,Entity F1
1,No log,0.102513,0.884120,0.907489,0.895652
2,No log,0.088032,0.883369,0.900881,0.892039
3,No log,0.086539,0.899782,0.909692,0.904710
4,No log,0.096513,0.900648,0.918502,0.909487
5,No log,0.126544,0.901376,0.865639,0.883146
6,No log,0.118494,0.898925,0.920705,0.909684
7,No log,0.115240,0.896018,0.892070,0.894040
8,No log,0.126576,0.892308,0.894273,0.893289
9,No log,0.127101,0.893333,0.885463,0.889381
10,No log,0.133692,0.894144,0.874449,0.884187


Kết quả trên Test Set của microsoft/mdeberta-v3-base:


{'eval_loss': 0.11584920436143875, 'eval_entity_precision': 0.9074074074073085, 'eval_entity_recall': 0.9174008810571677, 'eval_entity_f1': 0.9123767797965608, 'eval_runtime': 1.4719, 'eval_samples_per_second': 46.877, 'eval_steps_per_second': 6.114, 'epoch': 10.0}


#### No word segmentation

In [67]:
train_bio = create_bio(train, word_seg=False)
val_bio   = create_bio(val, word_seg=False)
test_bio  = create_bio(test, word_seg=False)

In [ ]:
model_name = "vinai/phobert-base"
tokenizer_name = "hieule/bartpho-tokenizer-fast"

res = run_experiment(model_name, tokenizer_name, train_bio, val_bio, test_bio,
                     lr=2e-5, epochs=10, save_path= "ner_phobert.pt")


Mô hình: vinai/phobert-base


Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Entity Precision,Entity Recall,Entity F1
1,No log,0.285964,0.520325,0.602353,0.558342
2,No log,0.206589,0.684435,0.755294,0.718121
3,No log,0.165894,0.737581,0.803529,0.769144
4,No log,0.147104,0.811343,0.824706,0.817970
5,No log,0.141825,0.795759,0.838824,0.816724
6,No log,0.138883,0.812570,0.851765,0.831706
7,No log,0.131097,0.832390,0.864706,0.848240
8,No log,0.131427,0.827273,0.856471,0.841618
9,No log,0.133612,0.827982,0.849412,0.838560
10,No log,0.134302,0.819635,0.844706,0.831981


Kết quả trên Test Set của vinai/phobert-base:


{'eval_loss': 0.11765109747648239, 'eval_entity_precision': 0.8418604651162301, 'eval_entity_recall': 0.8487690504102667, 'eval_entity_f1': 0.8453006420982293, 'eval_runtime': 1.1803, 'eval_samples_per_second': 65.236, 'eval_steps_per_second': 8.472, 'epoch': 10.0}


In [ ]:
tokenizer_name = "xlm-roberta-base"
model_name = "xlm-roberta-base"

res = run_experiment(model_name, tokenizer_name, train_bio, val_bio, test_bio,
                     lr=2e-5, epochs=10, save_dir= "ner_model")


Mô hình: xlm-roberta-base


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Entity Precision,Entity Recall,Entity F1
1,No log,0.119120,0.683371,0.835655,0.751880
2,No log,0.074374,0.837357,0.917827,0.875748
3,No log,0.065733,0.860104,0.924791,0.891275
4,No log,0.070782,0.855670,0.924791,0.888889
5,No log,0.073424,0.854194,0.922006,0.886805
6,No log,0.077923,0.859717,0.930362,0.893645
7,No log,0.081495,0.865285,0.930362,0.896644
8,No log,0.086981,0.863402,0.933148,0.896921
9,No log,0.086887,0.861183,0.933148,0.895722
10,No log,0.085988,0.858430,0.928969,0.892308


Saving best NER model to ner_model
Kết quả trên Test Set của xlm-roberta-base:


{'eval_loss': 0.08462507277727127, 'eval_entity_precision': 0.8899297423887067, 'eval_entity_recall': 0.9296636085626342, 'eval_entity_f1': 0.9093628476915189, 'eval_runtime': 1.2516, 'eval_samples_per_second': 63.916, 'eval_steps_per_second': 7.99, 'epoch': 10.0}


In [ ]:
tokenizer_name = "microsoft/mdeberta-v3-base"
model_name = "microsoft/mdeberta-v3-base"

res = run_experiment(model_name, tokenizer_name, train_bio, val_bio, test_bio,
                     lr=5e-5, epochs=10, save_dir= "ner_model")


Mô hình: microsoft/mdeberta-v3-base


Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/mdeberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Entity Precision,Entity Recall,Entity F1
1,No log,0.123462,0.859100,0.894094,0.876248
2,No log,0.102042,0.869732,0.924644,0.896347
3,No log,0.101788,0.873541,0.914460,0.893532
4,No log,0.113013,0.865759,0.906314,0.885572
5,No log,0.148474,0.862275,0.879837,0.870968
6,No log,0.132444,0.861272,0.910387,0.885149
7,No log,0.147026,0.865613,0.892057,0.878636
8,No log,0.155252,0.872510,0.892057,0.882175
9,No log,0.171880,0.871032,0.894094,0.882412
10,No log,0.167600,0.867589,0.894094,0.880642


Saving best NER model to ner_model
Kết quả trên Test Set của microsoft/mdeberta-v3-base:


{'eval_loss': 0.07324633002281189, 'eval_entity_precision': 0.8846880907371564, 'eval_entity_recall': 0.9294935451836217, 'eval_entity_f1': 0.9065375302162866, 'eval_runtime': 1.6751, 'eval_samples_per_second': 46.564, 'eval_steps_per_second': 5.97, 'epoch': 10.0}


# Coreference resolution

In [ ]:
def build_coref(raw_text):
    tokens, bio_tags, token_spans = build_bio_from_raw_text(raw_text)
    spans = bio_to_spans(bio_tags)

    gold_clusters = []
    for s, e in spans:
        eid = None
        for ts in token_spans:
            if not (e <= ts["start"] or s > ts["end"]):
                eid = ts["eid"]
                break
        gold_clusters.append(eid)

    return {
        "tokens": tokens,
        "bio_tags": bio_tags,
        "token_spans": token_spans,
        "gold_clusters": gold_clusters
    }

coref_train = [build_coref(t) for t in train]
coref_val   = [build_coref(t) for t in val]
coref_test  = [build_coref(t) for t in test]

## Mention representation

In [27]:
def get_mention_embeddings(hidden, word_ids, spans, gold_clusters):
    embs, kept_clusters, kept_spans = [], [], []

    for (s, e), cluster_id in zip(spans, gold_clusters):
        idx = [i for i, wid in enumerate(word_ids) if wid is not None and s <= wid < e]
        if not idx:
            continue

        emb = hidden[idx].mean(dim=0)
        embs.append(emb)
        kept_clusters.append(cluster_id)
        kept_spans.append((s, e))   # 👈 QUAN TRỌNG

    if not embs:
        return None, None, None

    return torch.stack(embs), kept_clusters, kept_spans

def get_test_mention_embeddings(hidden, word_ids, spans):
    embs = []
    kept_spans = []

    for s, e in spans:
        idx = [i for i, wid in enumerate(word_ids) if wid is not None and s <= wid < e]
        if not idx:
            continue

        embs.append(hidden[idx].mean(dim=0))
        kept_spans.append((s, e))

    if not embs:
        return None, None

    return torch.stack(embs), kept_spans

## Encode

In [ ]:
def encode_tokens(encoder, tokenizer, tokens, device, max_length=256):
    enc = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(device)

    with torch.no_grad():
        out = encoder(**enc)

    hidden = out.last_hidden_state[0]   # [T_sub, H]
    word_ids = enc.word_ids()           # subword -> token

    return hidden, word_ids

## Build pairs

In [29]:
def get_dist_bucket(dist):
    if dist <= 1: return 0
    if dist <= 2: return 1
    if dist <= 3: return 2
    if dist <= 4: return 3
    if dist <= 10: return 4
    if dist <= 20: return 5
    return 6

In [30]:
def build_coref_pairs_1(mention_embs, gold_clusters, spans, tokens, max_neg_per_pos=5):
    pairs = []
    N = len(gold_clusters)

    for i in range(N):
        for j in range(i + 1, N):
            if gold_clusters[i] is None:
                continue

            label = float(gold_clusters[i] == gold_clusters[j])

            same_string = float(
                tokens[spans[i][0]:spans[i][1]] ==
                tokens[spans[j][0]:spans[j][1]]
            )

            pairs.append({
                "mi": mention_embs[i],
                "mj": mention_embs[j],
                "dist": torch.tensor([j - i], dtype=torch.float),
                "same_string": torch.tensor([same_string], dtype=torch.float),
                "label": torch.tensor(label, dtype=torch.float)
            })

    return pairs

In [ ]:
def build_coref_pairs_2(mention_embs, gold_clusters, spans, tokens, max_neg_per_pos=None):
    pairs = []
    N = len(mention_embs) 

    for i in range(N):
        for j in range(i + 1, N):
            if gold_clusters[i] is None or gold_clusters[j] is None:
                label = 0.0
            else:
                label = float(gold_clusters[i] == gold_clusters[j])

            same_string = float(
                tokens[spans[i][0]:spans[i][1]] ==
                tokens[spans[j][0]:spans[j][1]]
            )

            pairs.append({
                "mi": mention_embs[i],
                "mj": mention_embs[j],
                "dist": torch.tensor([get_dist_bucket(j - i)], dtype=torch.long),
                "same_string": torch.tensor([same_string], dtype=torch.float),
                "label": torch.tensor(label, dtype=torch.float)
            })
    return pairs

## Scorer

In [33]:
class CorefScorer(nn.Module):
    def __init__(self, hidden_dim, dist_emb_dim=20, use_diff=True, use_dist=True, use_string=True):
        super().__init__()
        self.use_diff = use_diff
        self.use_dist = use_dist
        self.use_string = use_string

        # Tính toán input_dim
        input_dim = hidden_dim * 2
        if use_diff:
            input_dim += hidden_dim
        if use_string:
            input_dim += 1

        if use_dist:
            self.dist_embeddings = nn.Embedding(7, dist_emb_dim)
            input_dim += dist_emb_dim

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 1)
        )

    def forward(self, mi, mj, dist_ids, same_string):
        features = [mi, mj]

        if self.use_diff:
            features.append(torch.abs(mi - mj))

        if self.use_dist:
            dist_features = self.dist_embeddings(dist_ids.long().squeeze(-1))
            features.append(dist_features)

        if self.use_string:
            features.append(same_string if same_string.dim() > 1 else same_string.unsqueeze(-1))

        x = torch.cat(features, dim=-1)
        return self.mlp(x).squeeze(-1)

In [34]:
def compute_pairwise_scores(model, mention_embs):
    model.eval()
    device = mention_embs.device
    N = mention_embs.size(0)
    scores = torch.zeros(N, N, device=device)

    with torch.no_grad():
        for i in range(N):
            for j in range(i + 1, N):
                dist = torch.log(
                    torch.tensor([[j - i + 1]], device=device)
                )
                same_string = torch.zeros(1, 1, device=device)

                s = model(
                    mention_embs[i:i+1],
                    mention_embs[j:j+1],
                    dist,
                    same_string
                )
                scores[i, j] = torch.sigmoid(s)

    return scores

## Clustering

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        px, py = self.find(x), self.find(y)
        if px != py:
            self.parent[py] = px

def union_find_clustering(scores, threshold=0.5):
    N = scores.size(0)
    uf = UnionFind(N)

    for i in range(N):
        for j in range(i + 1, N):
            if scores[i, j] >= threshold:
                uf.union(i, j)

    clusters_dict = {}
    for i in range(N):
        root = uf.find(i)
        if root not in clusters_dict:
            clusters_dict[root] = set()
        clusters_dict[root].add(i)

    return list(clusters_dict.values())

In [36]:
def run_coref_document(tokens, bio_tags, encoder, tokenizer, model, device, max_length, threshold=0.5):
    spans = bio_to_spans(bio_tags)
    hidden, word_ids = encode_tokens(encoder, tokenizer, tokens, device, max_length)

    mention_embs, kept_spans = get_test_mention_embeddings(
        hidden, word_ids, spans
    )

    if mention_embs is None or mention_embs.size(0) < 2:
        return {"spans": kept_spans, "pred_clusters": []}

    scores = compute_pairwise_scores(model, mention_embs)
    pred_clusters = union_find_clustering(scores, threshold=threshold)

    return {
        "spans": kept_spans,
        "pred_clusters": pred_clusters
    }

## Evaluation

In [37]:
def pairwise_f1(pred_clusters, gold_clusters):
    def cluster_pairs(clusters):
        pairs = set()
        for c in clusters:
            c = list(c)
            for i in range(len(c)):
                for j in range(i + 1, len(c)):
                    pairs.add((c[i], c[j]))
        return pairs

    gold_pairs = set()
    N = len(gold_clusters)
    for i in range(N):
        for j in range(i + 1, N):
            if gold_clusters[i] is not None and gold_clusters[i] == gold_clusters[j]:
                gold_pairs.add((i, j))

    pred_pairs = cluster_pairs(pred_clusters)

    tp = len(pred_pairs & gold_pairs)
    fp = len(pred_pairs - gold_pairs)
    fn = len(gold_pairs - pred_pairs)

    p = tp / (tp + fp + 1e-8)
    r = tp / (tp + fn + 1e-8)
    f1 = 2 * p * r / (p + r + 1e-8)

    return p, r, f1

In [38]:
def evaluate_coref(dataset, encoder, tokenizer, model, device, max_length):
    ps, rs, fs = [], [], []

    for sample in dataset:
        out = run_coref_document(
            tokens=sample["tokens"],
            bio_tags=sample["bio_tags"],
            encoder=encoder,
            tokenizer=tokenizer,
            model=model,
            device=device,
            max_length=max_length
        )

        p, r, f1 = pairwise_f1(
            out["pred_clusters"],
            sample["gold_clusters"]
        )

        ps.append(p)
        rs.append(r)
        fs.append(f1)

    return sum(ps)/len(ps), sum(rs)/len(rs), sum(fs)/len(fs)

In [ ]:
def gold_clusters_from_ids(gold_clusters):
    clusters = {}
    for i, cid in enumerate(gold_clusters):
        if cid is None:
            continue
        if cid not in clusters:
            clusters[cid] = set()
        clusters[cid].add(i)
    return list(clusters.values())

In [ ]:
def muc_score(pred_clusters, gold_clusters):
    def links(clusters):
        return sum(len(c) - 1 for c in clusters if len(c) > 1)

    def correct(pred, gold):
        score = 0
        for p in pred:
            partitions = sum(1 for g in gold if len(p & g) > 0)
            if len(p) > 0:
                score += len(p) - partitions
        return score

    recall = correct(gold_clusters, pred_clusters) / max(links(gold_clusters), 1e-8)
    precision = correct(pred_clusters, gold_clusters) / max(links(pred_clusters), 1e-8)

    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1

In [ ]:
def b_cubed_score(pred_clusters, gold_clusters, N):
    gold_map = {}
    pred_map = {}

    for c in gold_clusters:
        for m in c:
            gold_map[m] = c

    for c in pred_clusters:
        for m in c:
            pred_map[m] = c

    p_sum = r_sum = 0.0
    for i in range(N):
        if i not in gold_map or i not in pred_map:
            continue

        inter = len(gold_map[i] & pred_map[i])
        p_sum += inter / len(pred_map[i])
        r_sum += inter / len(gold_map[i])

    precision = p_sum / N
    recall = r_sum / N
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1

In [ ]:
def ceaf_phi4(pred_clusters, gold_clusters):
    def phi4(c1, c2):
        return 2 * len(c1 & c2) / (len(c1) + len(c2) + 1e-8)

    sim = np.zeros((len(gold_clusters), len(pred_clusters)))
    for i, g in enumerate(gold_clusters):
        for j, p in enumerate(pred_clusters):
            sim[i, j] = phi4(g, p)

    row, col = linear_sum_assignment(-sim)
    score = sim[row, col].sum()

    precision = score / max(len(pred_clusters), 1e-8)
    recall = score / max(len(gold_clusters), 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1

In [ ]:
def evaluate_conll(dataset, encoder, tokenizer, model, device, max_length):
    muc_p = muc_r = muc_f = 0
    b3_p = b3_r = b3_f = 0
    ceaf_p = ceaf_r = ceaf_f = 0
    n_docs = 0

    for sample in dataset:
        out = run_coref_document(
            tokens=sample["tokens"],
            bio_tags=sample["bio_tags"],
            encoder=encoder,
            tokenizer=tokenizer,
            model=model,
            device=device,
            max_length=max_length
        )

        pred = out["pred_clusters"]
        gold = gold_clusters_from_ids(sample["gold_clusters"])

        if len(pred) == 0 or len(gold) == 0:
            continue

        n = sum(len(c) for c in gold)

        p, r, f = muc_score(pred, gold)
        muc_p += p; muc_r += r; muc_f += f

        p, r, f = b_cubed_score(pred, gold, n)
        b3_p += p; b3_r += r; b3_f += f

        p, r, f = ceaf_phi4(pred, gold)
        ceaf_p += p; ceaf_r += r; ceaf_f += f

        n_docs += 1

    return {
        "MUC":   (muc_p/n_docs,  muc_r/n_docs,  muc_f/n_docs),
        "B3":    (b3_p/n_docs,   b3_r/n_docs,   b3_f/n_docs),
        "CEAF":  (ceaf_p/n_docs, ceaf_r/n_docs, ceaf_f/n_docs),
        "CoNLL_F1": (
            (muc_f + b3_f + ceaf_f) / (3 * n_docs)
        )
    }

## Training

In [ ]:
def train_coref_epoch(model, dataset, encoder, tokenizer, optimizer, device, max_length):
    model.train()
    loss_fn = nn.BCEWithLogitsLoss()
    total_loss, steps = 0.0, 0

    loop = tqdm(dataset, desc="Train coref", leave=False)

    for sample in loop:
        # Trích xuất mention và tính embedding
        spans = bio_to_spans(sample["bio_tags"])
        hidden, word_ids = encode_tokens(encoder, tokenizer, sample["tokens"], device, max_length)

        mention_embs, clusters, kept_spans = get_mention_embeddings(
            hidden, word_ids, spans, sample["gold_clusters"]
        )

        if mention_embs is None or len(clusters) < 2:
            continue

        # Xây dựng cặp 
        pairs = build_coref_pairs_2(mention_embs, clusters, kept_spans, sample["tokens"])
        if not pairs:
            continue

        # Gom nhóm (Batching) các cặp trong 1 document 
        batch_mi = torch.stack([p["mi"] for p in pairs]).to(device)
        batch_mj = torch.stack([p["mj"] for p in pairs]).to(device)
        batch_dist = torch.stack([p["dist"] for p in pairs]).to(device)
        batch_same_str = torch.stack([p["same_string"] for p in pairs]).to(device)
        batch_labels = torch.stack([p["label"] for p in pairs]).to(device)

        optimizer.zero_grad()

        # Forward pass
        scores = model(
            mi=batch_mi,
            mj=batch_mj,
            dist_ids=batch_dist, 
            same_string=batch_same_str
        )

        # Tính Loss và Backward
        loss = loss_fn(scores, batch_labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        steps += 1

        loop.set_postfix(
            loss=f"{total_loss/steps:.4f}",
            pairs=len(pairs)
        )

    return total_loss / max(steps, 1)

In [ ]:
def run_coref_experiment(
    model_name,
    token_name,
    train_data,
    val_data,
    test_data,
    lr=1e-5,
    epochs=10,
    save_path="best_model.pt",
    use_dist=True,
    use_diff=True,
    use_string=True,
    max_length=256,
    device="cuda"
):
    print(f"========  Model: {model_name}  =======")

    # Load Encoder & tokenizer
    tokenizer = AutoTokenizer.from_pretrained(token_name, use_fast=True)
    encoder = AutoModel.from_pretrained(model_name).to(device)
    encoder.eval()
    for p in encoder.parameters():
        p.requires_grad = False

    # Khởi tạo CorefScorer
    hidden_dim = encoder.config.hidden_size
    coref_model = CorefScorer(
        hidden_dim=hidden_dim,
        use_diff=use_diff,
        use_dist=use_dist,
        use_string=use_string
    ).to(device)

    optimizer = torch.optim.AdamW(coref_model.parameters(), lr=lr)

    best_val_f1 = 0.0
    best_epoch = -1

    # Training Loop
    for epoch in range(epochs):
        train_loss = train_coref_epoch(
            coref_model, train_data, encoder, tokenizer, optimizer, device, max_length
        )

        val_scores = evaluate_conll(val_data, encoder, tokenizer, coref_model, device, max_length)
        val_f1 = float(val_scores["CoNLL_F1"])

        print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val F1: {val_f1:.4f}")

        # Lưu Best Model
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch + 1
            checkpoint = {
                "coref_state_dict": coref_model.state_dict(),
                "coref_config": {
                    "hidden_dim": hidden_dim,
                    "use_dist": use_dist,
                    "use_diff": use_diff,
                    "use_string": use_string
                },
                "encoder_name": model_name, 
                "token_name": token_name
            }
            torch.save(checkpoint, save_path)

    # Load lại bản tốt nhất 
    checkpoint = torch.load(save_path)
    coref_model.load_state_dict(checkpoint["coref_state_dict"])

    print(f"--- Final Evaluation (Best Epoch {best_epoch}) ---")
    scores = evaluate_conll(test_data, encoder, tokenizer, coref_model, device, max_length)

    for k in ["MUC", "B3", "CEAF"]:
        p, r, f = scores[k]
        print(f"{k:5s} | P={p:.4f} R={r:.4f} F1={f:.4f}")
    print(f" CoNLL F1 = {scores['CoNLL_F1']:.4f}")

    return scores

# Result coref

#### Full model

In [42]:
results_full = {}

In [ ]:
model_name = "vinai/phobert-base"
token_name = "hieule/bartpho-tokenizer-fast"

results_full["phobert"] = run_coref_experiment(
    model_name, token_name, coref_train, coref_val, coref_test,lr=0.01, epochs=20,
    save_path="coref_phobert_full.pt", use_dist=True, use_diff=True, use_string=True, device=device
)

========  Model: vinai/phobert-base  =======


Epoch 01 | Train Loss: 0.5071 | Val F1: 0.2298


Epoch 02 | Train Loss: 0.3358 | Val F1: 0.2398


Epoch 03 | Train Loss: 0.2839 | Val F1: 0.2528


Epoch 04 | Train Loss: 0.2597 | Val F1: 0.2611


...

Epoch 16 | Train Loss: 0.1831 | Val F1: 0.2594


Epoch 17 | Train Loss: 0.1836 | Val F1: 0.2432


Epoch 18 | Train Loss: 0.1757 | Val F1: 0.2483


Epoch 19 | Train Loss: 0.1748 | Val F1: 0.2464


Epoch 20 | Train Loss: 0.1677 | Val F1: 0.2617
--- Final Evaluation (Best Epoch 11) ---
MUC   | P=0.9395 R=0.9903 F1=0.9608
B3    | P=0.4988 R=0.3624 F1=0.4020
CEAF  | P=0.5470 R=0.4961 F1=0.5053
 CoNLL F1 = 0.6227


In [ ]:
model_name = "xlm-roberta-base"
token_name = "xlm-roberta-base"

results_full["roberta"] = run_coref_experiment(
    model_name, token_name, coref_train, coref_val, coref_test, lr=0.002, epochs=20,
    save_path="coref_roberta_full.pt", use_dist=True, use_diff=True, use_string=True,
    max_length=512, device=device
)

========  Model: xlm-roberta-base  =======


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

Epoch 16 | Train Loss: 0.1164 | Val F1: 0.4959


Epoch 17 | Train Loss: 0.1020 | Val F1: 0.5168


Epoch 18 | Train Loss: 0.0945 | Val F1: 0.5313


Epoch 19 | Train Loss: 0.0859 | Val F1: 0.5290


Epoch 20 | Train Loss: 0.0811 | Val F1: 0.5276
--- Final Evaluation (Best Epoch 18) ---
MUC   | P=0.9304 R=0.9628 F1=0.9433
B3    | P=0.6975 R=0.6810 F1=0.6764
CEAF  | P=0.6978 R=0.6579 F1=0.6608
 CoNLL F1 = 0.7602


In [ ]:
model_name = "microsoft/mdeberta-v3-base"
token_name = "microsoft/mdeberta-v3-base"

results_full["mdeberta"] = run_coref_experiment(
    model_name, token_name, coref_train, coref_val, coref_test, lr=0.002, epochs=20,
    save_path="coref_mdeberta_full.pt", use_dist=True, use_diff=True, use_string=True,
    max_length=512, device=device
)

========  Model: microsoft/mdeberta-v3-base  =======


Epoch 01 | Train Loss: 0.4624 | Val F1: 0.6239


Epoch 02 | Train Loss: 0.3390 | Val F1: 0.6569


Epoch 03 | Train Loss: 0.3086 | Val F1: 0.6738


Epoch 04 | Train Loss: 0.2690 | Val F1: 0.6785


...

Epoch 16 | Train Loss: 0.1551 | Val F1: 0.6891


Epoch 17 | Train Loss: 0.1529 | Val F1: 0.6840


Epoch 18 | Train Loss: 0.1502 | Val F1: 0.6778


Epoch 19 | Train Loss: 0.1495 | Val F1: 0.6850


Epoch 20 | Train Loss: 0.1560 | Val F1: 0.6797
--- Final Evaluation (Best Epoch 15) ---
MUC   | P=0.9232 R=0.9480 F1=0.9315
B3    | P=0.5619 R=0.4827 F1=0.5060
CEAF  | P=0.5858 R=0.5867 F1=0.5716
 CoNLL F1 = 0.6697


#### No diff

In [53]:
results_no_diff = {}

In [ ]:
model_name = "vinai/phobert-base"
token_name = "hieule/bartpho-tokenizer-fast"

results_no_diff["phobert"] = run_coref_experiment(
    model_name, token_name, coref_train, coref_val, coref_test, lr = 0.01, epochs=20,
    save_path="coref_phobert_diff.pt", use_dist=True, use_diff=False, use_string=True
)

========  Model: vinai/phobert-base  =======


Epoch 01 | Train Loss: 0.4859 | Val F1: 0.5410


Epoch 02 | Train Loss: 0.3447 | Val F1: 0.5706


Epoch 03 | Train Loss: 0.3039 | Val F1: 0.5741


Epoch 04 | Train Loss: 0.2722 | Val F1: 0.5738


...

Epoch 16 | Train Loss: 0.1690 | Val F1: 0.5478


Epoch 17 | Train Loss: 0.1603 | Val F1: 0.5675


Epoch 18 | Train Loss: 0.1494 | Val F1: 0.5404


Epoch 19 | Train Loss: 0.1546 | Val F1: 0.4858


Epoch 20 | Train Loss: 0.1651 | Val F1: 0.4850
--- Final Evaluation (Best Epoch 11) ---
MUC   | P=0.8788 R=0.9260 F1=0.8781
B3    | P=0.4748 R=0.3372 F1=0.3713
CEAF  | P=0.4544 R=0.4617 F1=0.4262
 CoNLL F1 = 0.5585


In [ ]:
model_name = "xlm-roberta-base"
token_name = "xlm-roberta-base"

results_no_diff["roberta"] = run_coref_experiment(
    model_name, token_name, coref_train, coref_val, coref_test, lr=0.002, epochs=20,
    save_path="coref_roberta_diff.pt", use_dist=True, use_diff=False, use_string=True,
    max_length=512, device=device
)

========  Model: xlm-roberta-base  =======


Epoch 01 | Train Loss: 0.3655 | Val F1: 0.6089


Epoch 02 | Train Loss: 0.2470 | Val F1: 0.6351


Epoch 03 | Train Loss: 0.2020 | Val F1: 0.6208


Epoch 04 | Train Loss: 0.1759 | Val F1: 0.6353


...

Epoch 16 | Train Loss: 0.0828 | Val F1: 0.6862


Epoch 17 | Train Loss: 0.0804 | Val F1: 0.6763


Epoch 18 | Train Loss: 0.0782 | Val F1: 0.6886


Epoch 19 | Train Loss: 0.0732 | Val F1: 0.6590


Epoch 20 | Train Loss: 0.0710 | Val F1: 0.6635
--- Final Evaluation (Best Epoch 11) ---
MUC   | P=0.9111 R=0.8992 F1=0.8988
B3    | P=0.7079 R=0.6246 F1=0.6450
CEAF  | P=0.5931 R=0.6336 F1=0.5896
 CoNLL F1 = 0.7112


In [ ]:
model_name = "microsoft/mdeberta-v3-base"
token_name = "microsoft/mdeberta-v3-base"

results_no_diff["mdeberta"] = run_coref_experiment(
    model_name, token_name, coref_train, coref_val, coref_test, lr=0.002, epochs=20,
    save_path="coref_mdeberta_diff.pt", use_dist=True, use_diff=False, use_string=True,
    max_length=512, device=device
)

========  Model: microsoft/mdeberta-v3-base  =======


Epoch 01 | Train Loss: 0.4677 | Val F1: 0.6342


Epoch 02 | Train Loss: 0.3175 | Val F1: 0.6424


Epoch 03 | Train Loss: 0.2769 | Val F1: 0.6558


Epoch 04 | Train Loss: 0.2517 | Val F1: 0.6872


...

Epoch 16 | Train Loss: 0.1119 | Val F1: 0.6992


Epoch 17 | Train Loss: 0.1069 | Val F1: 0.7011


Epoch 18 | Train Loss: 0.1021 | Val F1: 0.6970


Epoch 19 | Train Loss: 0.0981 | Val F1: 0.6975


Epoch 20 | Train Loss: 0.0915 | Val F1: 0.7036
--- Final Evaluation (Best Epoch 20) ---
MUC   | P=0.9288 R=0.9490 F1=0.9334
B3    | P=0.5802 R=0.4852 F1=0.5147
CEAF  | P=0.5918 R=0.5906 F1=0.5745
 CoNLL F1 = 0.6742


#### No dist

In [43]:
results_no_dist = {}

In [45]:
model_name = "vinai/phobert-base"
token_name = "hieule/bartpho-tokenizer-fast"

results_no_dist["phobert"] = run_coref_experiment(
    model_name, token_name, coref_train, coref_val, coref_test, lr=0.01, epochs=20,
    save_path="coref_phobert_dist.pt", use_dist=False, use_diff=True, use_string=True
)

========  Model: vinai/phobert-base  =======


Epoch 01 | Train Loss: 0.5505 | Val F1: 0.5233


Epoch 02 | Train Loss: 0.3267 | Val F1: 0.5434


Epoch 03 | Train Loss: 0.2771 | Val F1: 0.5555


Epoch 04 | Train Loss: 0.2589 | Val F1: 0.5750


...

Epoch 16 | Train Loss: 0.2341 | Val F1: 0.5777


Epoch 17 | Train Loss: 0.2225 | Val F1: 0.5382


Epoch 18 | Train Loss: 0.2204 | Val F1: 0.5767


Epoch 19 | Train Loss: 0.2237 | Val F1: 0.5851


Epoch 20 | Train Loss: 0.2334 | Val F1: 0.5656
--- Final Evaluation (Best Epoch 6) ---
MUC   | P=0.9159 R=0.9465 F1=0.9254
B3    | P=0.4907 R=0.3624 F1=0.3967
CEAF  | P=0.5187 R=0.4978 F1=0.4857
 CoNLL F1 = 0.6026


In [ ]:
model_name = "xlm-roberta-base"
token_name = "xlm-roberta-base"

results_no_dist["roberta"] = run_coref_experiment(
    model_name, token_name, coref_train, coref_val, coref_test, lr=0.002, epochs=20,
    save_path="coref_roberta_dist.pt", use_dist=False, use_diff=True, use_string=True,
    max_length=512, device=device
)

========  Model: xlm-roberta-base  =======


Epoch 01 | Train Loss: 0.3332 | Val F1: 0.7278


Epoch 02 | Train Loss: 0.2222 | Val F1: 0.7345


Epoch 03 | Train Loss: 0.1956 | Val F1: 0.7364


Epoch 04 | Train Loss: 0.1801 | Val F1: 0.7447


...

Epoch 16 | Train Loss: 0.1138 | Val F1: 0.7287


Epoch 17 | Train Loss: 0.1092 | Val F1: 0.7293


Epoch 18 | Train Loss: 0.1060 | Val F1: 0.7393


Epoch 19 | Train Loss: 0.1027 | Val F1: 0.7145


Epoch 20 | Train Loss: 0.1016 | Val F1: 0.7225
--- Final Evaluation (Best Epoch 9) ---
MUC   | P=0.9504 R=0.9223 F1=0.9336
B3    | P=0.6995 R=0.6553 F1=0.6603
CEAF  | P=0.5956 R=0.6801 F1=0.6225
 CoNLL F1 = 0.7388


In [ ]:
model_name = "microsoft/mdeberta-v3-base"
token_name = "microsoft/mdeberta-v3-base"

results_no_dist["mdeberta"] = run_coref_experiment(
    model_name, token_name, coref_train, coref_val, coref_test, lr=0.002, epochs=20,
    save_path="coref_mdeberta_dist.pt", use_dist=False, use_diff=True, use_string=True,
    max_length=512, device=device
)

========  Model: microsoft/mdeberta-v3-base  =======


Epoch 01 | Train Loss: 0.4593 | Val F1: 0.6496


Epoch 02 | Train Loss: 0.3282 | Val F1: 0.6562


Epoch 03 | Train Loss: 0.2902 | Val F1: 0.6496


Epoch 04 | Train Loss: 0.2661 | Val F1: 0.6520


...

Epoch 16 | Train Loss: 0.1451 | Val F1: 0.6902


Epoch 17 | Train Loss: 0.1432 | Val F1: 0.6766


Epoch 18 | Train Loss: 0.1405 | Val F1: 0.6786


Epoch 19 | Train Loss: 0.1365 | Val F1: 0.6771


Epoch 20 | Train Loss: 0.1404 | Val F1: 0.6792
--- Final Evaluation (Best Epoch 16) ---
MUC   | P=0.9114 R=0.9480 F1=0.9243
B3    | P=0.5571 R=0.4762 F1=0.5005
CEAF  | P=0.5725 R=0.5654 F1=0.5528
 CoNLL F1 = 0.6592


#### No same_string

In [47]:
results_no_string = {}

In [49]:
model_name = "vinai/phobert-base"
token_name = "hieule/bartpho-tokenizer-fast"

results_no_string["phobert"] = run_coref_experiment(
    model_name, token_name, coref_train, coref_val, coref_test, lr=0.01, epochs=20,
    save_path="coref_phobert_string.pt", use_dist=True, use_diff=True, use_string=False
)

========  Model: vinai/phobert-base  =======


Epoch 01 | Train Loss: 0.5789 | Val F1: 0.5277


Epoch 02 | Train Loss: 0.3424 | Val F1: 0.5616


Epoch 03 | Train Loss: 0.3134 | Val F1: 0.5743


Epoch 04 | Train Loss: 0.2876 | Val F1: 0.5910


...

Epoch 16 | Train Loss: 0.2112 | Val F1: 0.5850


Epoch 17 | Train Loss: 0.2028 | Val F1: 0.5790


Epoch 18 | Train Loss: 0.2269 | Val F1: 0.6010


Epoch 19 | Train Loss: 0.2174 | Val F1: 0.5998


Epoch 20 | Train Loss: 0.2261 | Val F1: 0.5090
--- Final Evaluation (Best Epoch 13) ---
MUC   | P=0.9059 R=0.9542 F1=0.9255
B3    | P=0.4831 R=0.3641 F1=0.3954
CEAF  | P=0.5373 R=0.5008 F1=0.5048
 CoNLL F1 = 0.6086


In [ ]:
model_name = "xlm-roberta-base"
token_name = "xlm-roberta-base"

results_no_string["roberta"] = run_coref_experiment(
    model_name, token_name, coref_train, coref_val, coref_test, lr=0.002, epochs=20,
    save_path="coref_roberta_string.pt", use_dist=True, use_diff=True, use_string=False,
    max_length=512, device=device
)

========  Model: xlm-roberta-base  =======


Epoch 01 | Train Loss: 0.3395 | Val F1: 0.7311


Epoch 02 | Train Loss: 0.2248 | Val F1: 0.7372


Epoch 03 | Train Loss: 0.1991 | Val F1: 0.7460


Epoch 04 | Train Loss: 0.1831 | Val F1: 0.7471


...

Epoch 16 | Train Loss: 0.1177 | Val F1: 0.7315


Epoch 17 | Train Loss: 0.1137 | Val F1: 0.7331


Epoch 18 | Train Loss: 0.1087 | Val F1: 0.7234


Epoch 19 | Train Loss: 0.1052 | Val F1: 0.7271


Epoch 20 | Train Loss: 0.1012 | Val F1: 0.7286
--- Final Evaluation (Best Epoch 7) ---
MUC   | P=0.9526 R=0.9294 F1=0.9379
B3    | P=0.7264 R=0.6571 F1=0.6777
CEAF  | P=0.6292 R=0.7047 F1=0.6495
 CoNLL F1 = 0.7550


In [ ]:
model_name = "microsoft/mdeberta-v3-base"
token_name = "microsoft/mdeberta-v3-base"

results_no_string["mdeberta"] = run_coref_experiment(
    model_name, token_name, coref_train, coref_val, coref_test, lr=0.002, epochs=20,
    save_path="coref_mdeberta_string.pt", use_dist=True, use_diff=True, use_string=False,
    max_length=512, device=device
)

========  Model: microsoft/mdeberta-v3-base  =======


Epoch 01 | Train Loss: 0.4962 | Val F1: 0.6300


Epoch 02 | Train Loss: 0.3589 | Val F1: 0.6406


Epoch 03 | Train Loss: 0.3081 | Val F1: 0.6643


Epoch 04 | Train Loss: 0.2831 | Val F1: 0.6687


...

Epoch 16 | Train Loss: 0.1910 | Val F1: 0.6825


Epoch 17 | Train Loss: 0.1830 | Val F1: 0.6809


Epoch 18 | Train Loss: 0.1800 | Val F1: 0.6738


Epoch 19 | Train Loss: 0.1760 | Val F1: 0.6622


Epoch 20 | Train Loss: 0.1714 | Val F1: 0.6513
--- Final Evaluation (Best Epoch 13) ---
MUC   | P=0.9149 R=0.9350 F1=0.9199
B3    | P=0.5467 R=0.4697 F1=0.4929
CEAF  | P=0.5540 R=0.5673 F1=0.5456
 CoNLL F1 = 0.6528


# Inference

## Utils

In [ ]:
def gold_eids_to_span_clusters(spans, gold_eids):
    clusters = defaultdict(list)

    for span, eid in zip(spans, gold_eids):
        if eid is None:
            continue
        clusters[eid].append(span)

    return list(clusters.values())

def clusters_from_ids_to_spans(pred_clusters, spans):
    span_clusters = []
    for cluster in pred_clusters:
        span_clusters.append(
            [spans[mid] for mid in cluster]
        )
    return span_clusters

def build_coref_instance(tokens, clusters):
    return {
        "tokens": tokens,
        "clusters": clusters
    }

def build_mention_to_cluster(clusters):

    m2c = {}
    for cid, cluster in enumerate(clusters):
        for m in cluster:
            m2c[tuple(m)] = cid
    return m2c

In [ ]:
def normalize_clusters(clusters):
    norm = []

    # GOLD: dict
    if isinstance(clusters, dict):
        for c in clusters.values():
            if isinstance(c, dict):
                norm.append(set(tuple(m) for m in c.keys()))
            else:
                norm.append(set(tuple(m) for m in c))

    # PRED: list
    elif isinstance(clusters, list):
        for c in clusters:
            norm.append(set(tuple(m) for m in c))

    else:
        raise TypeError(f"Unsupported cluster type: {type(clusters)}")

    return norm

In [ ]:
def normalize(clusters):
    return [set(tuple(m) for m in c) for c in clusters if c]

def muc_score_dataset(gold_list, pred_list):
    num_p, den_p, num_r, den_r = 0, 0, 0, 0

    def get_links_and_correct(c_src, c_target):
        links, correct = 0, 0
        target_map = {m: i for i, c in enumerate(c_target) for m in c}
        for cluster in c_src:
            if len(cluster) < 2: continue
            links += len(cluster) - 1
            # Tìm số cluster ở target mà cluster hiện tại giao thoa
            found_targets = set(target_map[m] for m in cluster if m in target_map)
            correct += len(cluster) - max(len(found_targets), 1) if found_targets else 0
        return links, correct

    for g, p in zip(gold_list, pred_list):
        g, p = normalize(g), normalize(p)
        d_p, n_p = get_links_and_correct(p, g)
        d_r, n_r = get_links_and_correct(g, p)
        num_p += n_p; den_p += d_p
        num_r += n_r; den_r += d_r

    p = num_p / den_p if den_p > 0 else 0.0
    r = num_r / den_r if den_r > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1


def b_cubed_score_dataset(gold_list, pred_list):
    p_num, p_den, r_num, r_den = 0.0, 0.0, 0.0, 0.0

    for g, p in zip(gold_list, pred_list):
        g, p = normalize(g), normalize(p)
        gold_map = {m: c for c in g for m in c}
        pred_map = {m: c for c in p for m in c}

        for m in pred_map:
            if m in gold_map:
                p_num += len(gold_map[m] & pred_map[m]) / len(pred_map[m])
        p_den += len(pred_map)

        for m in gold_map:
            if m in pred_map:
                r_num += len(gold_map[m] & pred_map[m]) / len(gold_map[m])
        r_den += len(gold_map)

    p = p_num / p_den if p_den > 0 else 0.0
    r = r_num / r_den if r_den > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1


def ceaf_score_dataset(gold_list, pred_list):
    sim_sum, den_g, den_p = 0.0, 0.0, 0.0

    for g, p in zip(gold_list, pred_list):
        g, p = normalize(g), normalize(p)
        if not g or not p:
            den_g += len(g); den_p += len(p)
            continue

        sim_matrix = np.zeros((len(g), len(p)))
        for i, g_c in enumerate(g):
            for j, p_c in enumerate(p):
                inter = len(g_c & p_c)
                sim_matrix[i, j] = 2 * inter / (len(g_c) + len(p_c))

        row, col = linear_sum_assignment(-sim_matrix)
        sim_sum += sim_matrix[row, col].sum()
        den_g += len(g); den_p += len(p)

    p = sim_sum / den_p if den_p > 0 else 0.0
    r = sim_sum / den_g if den_g > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1

In [ ]:
def eval_inference(gold_instances, pred_instances):
    all_gold_clusters = [doc["clusters"] for doc in gold_instances]
    all_pred_clusters = [doc["clusters"] for doc in pred_instances]

    m_p, m_r, m_f = muc_score_dataset(all_gold_clusters, all_pred_clusters)
    b_p, b_r, b_f = b_cubed_score_dataset(all_gold_clusters, all_pred_clusters)
    c_p, c_r, c_f = ceaf_score_dataset(all_gold_clusters, all_pred_clusters)

    conll_f1 = (m_f + b_f + c_f) / 3
    return {
        "MUC": {"p": m_p, "r": m_r, "f1": m_f},
        "B3":  {"p": b_p, "r": b_r, "f1": b_f},
        "CEAF": {"p": c_p, "r": c_r, "f1": c_f},
        "CoNLL_F1": conll_f1
    }

## Predict ner

In [63]:
def predict_bio(raw_text, ner_model, tokenizer, max_length, device):
    ner_model.eval()
    ner_model.to(device)

    # clean + tokenize
    clean_text = remove_M_tags(raw_text)
    tokens = tokenize(clean_text)

    # encode
    enc = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(device)

    with torch.no_grad():
        out = ner_model(**enc)

    preds = out.logits.argmax(-1)[0].cpu().tolist()
    word_ids = enc.word_ids()

    # recover token-level BIO
    bio_tags = ["O"] * len(tokens)
    prev_word_id = None

    for p, wid in zip(preds, word_ids):
        if wid is None:
            continue
        if wid != prev_word_id:
            bio_tags[wid] = id2label[p]
        prev_word_id = wid

    return tokens, bio_tags

## Predict coref

In [64]:
def predict_coref(tokens, bio_tags, encoder, tokenizer, coref_model,
                  device, threshold=0.5):
    spans = bio_to_spans(bio_tags)

    if len(spans) < 2:
        return {
            "spans": spans,
            "pred_clusters": []
        }

    hidden, word_ids = encode_tokens(
        encoder, tokenizer, tokens, device
    )

    mention_embs, kept_spans = get_test_mention_embeddings(
        hidden, word_ids, spans
    )

    if mention_embs is None or mention_embs.size(0) < 2:
        return {
            "spans": kept_spans,
            "pred_clusters": []
        }

    scores = compute_pairwise_scores(
        coref_model, mention_embs
    )

    pred_clusters = union_find_clustering(
        scores, threshold=threshold
    )

    return {
        "spans": kept_spans,
        "pred_clusters": pred_clusters
    }


## Predict inference

In [65]:
def run_inference(model_name, test_data, max_length, device, coref_path, ner_dir="/content/ner_model"):
    print(f"========  Model: {model_name}  =======")

    # Load ner
    ner_model = AutoModelForTokenClassification.from_pretrained(ner_dir)
    tokenizer = AutoTokenizer.from_pretrained(ner_dir)
    ner_model.to(device)
    ner_model.eval()

    # Load coref
    ckpt = torch.load(coref_path)

    # Khởi tạo lại Encoder từ name đã lưu
    encoder = AutoModel.from_pretrained(ckpt["encoder_name"]).to(device)
    coref_tokenizer = AutoTokenizer.from_pretrained(ckpt["token_name"])

    # Khởi tạo lại CorefScorer từ config đã lưu
    coref_model = CorefScorer(**ckpt["coref_config"]).to(device)
    coref_model.load_state_dict(ckpt["coref_state_dict"])

    encoder.eval()
    coref_model.eval()

    for p in encoder.parameters():
        p.requires_grad = False

    gold_instances = []
    pred_instances = []

    for raw_text in test:
        # ----- GOLD -----
        gold = build_coref(raw_text)
        spans = bio_to_spans(gold["bio_tags"])

        gold_clusters = gold_eids_to_span_clusters(
            spans, gold["gold_clusters"]
        )

        gold_instances.append({
            "tokens": gold["tokens"],
            "clusters": gold_clusters
        })

        # ----- PRED -----
        tokens, bio_tags = predict_bio(
            raw_text,
            ner_model,
            tokenizer,
            max_length,
            device
        )

        out = predict_coref(
            tokens,
            bio_tags,
            encoder,
            tokenizer,
            coref_model,
            device
        )

        pred_clusters = clusters_from_ids_to_spans(
            out["pred_clusters"], out["spans"]
        )

        pred_instances.append({
            "tokens": tokens,
            "clusters": pred_clusters
        })

    # Evaluate
    scores = eval_inference(gold_instances, pred_instances)
    for k, v in scores.items():
        print(k, v)

# Result Inference

In [ ]:
model_name = "phobert"

run_inference(model_name, test_data=test, max_length=256, device=device,
              coref_path="/content/coref_phobert_full.pt")

========  Model: phobert  =======
MUC {'p': 0.9306759098786829, 'r': 0.8591172536781096, 'f1': 0.8934660697556953}
B3 {'p': 0.7057008213795981, 'r': 0.15110952640641837, 'f1': 0.2489188352564663}
CEAF {'p': np.float64(0.32665428269039287), 'r': np.float64(0.3185049034964537), 'f1': np.float64(0.3225281233300931)}
CoNLL_F1 0.4883043427807516


In [ ]:
model_name = "roberta"

run_inference(model_name, test_data=test, max_length=512, device=device,
              coref_path="/content/coref_roberta_full.pt")

========  Model: roberta  =======


The tokenizer you are loading from '/content/ner_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


MUC {'p': 0.96274217585693, 'r': 0.8932291666666666, 'f1': 0.926683911278771}
B3 {'p': 0.7940953325679562, 'r': 0.1844977441500732, 'f1': 0.29942741469262396}
CEAF {'p': np.float64(0.45069504106347497), 'r': np.float64(0.3738949892230622), 'f1': np.float64(0.4087185421408964)}
CoNLL_F1 0.5449432893707638


In [ ]:
model_name = "mdeberta"

run_inference(model_name, test_data=test, max_length=512, device=device,
              coref_path="/content/coref_mdeberta_diff.pt")

========  Model: mdeberta  =======


The tokenizer you are loading from '/content/ner_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


MUC {'p': 0.9414519906323185, 'r': 0.8862366683595734, 'f1': 0.9130102892455226}
B3 {'p': 0.7633476438684763, 'r': 0.12402835126916788, 'f1': 0.21338587077628887}
CEAF {'p': np.float64(0.4643320935159254), 'r': np.float64(0.30710903702069103), 'f1': np.float64(0.3696991940221906)}
CoNLL_F1 0.49869845134800067
